# 06 — Evaluate a finished run

Everything below was already written by training. This notebook reads it back and
adds the analyses that need judgement: per-cohort breakdown, the source probe, and
the mistakes worth looking at.

**Responsibility:** interpret one run. Trains nothing.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dataset_config as config
from dataset_config import Config, TASKS

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})

import json
from core.experiment import load_experiments
from core import evaluation as ev

In [ ]:
runs = load_experiments()
if not runs:
    raise SystemExit("no finished runs in results/ — train one with notebook 04 or 05")
print(f"{len(runs)} finished runs\n")
print(pd.DataFrame(runs)[["run", "pipeline", "task", "model", "test_auc"]].to_string(index=False))

In [ ]:
# Pick one. Defaults to the most recent.
RUN = sorted(config.RESULTS_DIR.glob("test_*"))[-1]
print("inspecting:", RUN.name)

meta = json.loads((RUN / "results.json").read_text())
cfg  = meta["config"]
print(f"\n{cfg['pipeline']} / {cfg['task']} / {cfg['model']}  seed {cfg['seed']}")
print(f"best epoch {meta['best_epoch']} of {meta['epochs_run']} run")
print(f"trainable parameters: {meta['parameters']['trainable']:,} "
      f"({100*meta['parameters']['trainable']/meta['parameters']['total']:.1f}%)")

## Headline metrics

In [ ]:
for split, m in meta["splits"].items():
    print(f"--- {split} ---")
    print(f"  macro-AUC          {m['auc']:.4f}")
    print(f"  accuracy           {m['accuracy']:.4f}   (trivial "
          f"{m['trivial_baseline_accuracy']:.4f})")
    print(f"  balanced accuracy  {m['balanced_accuracy']:.4f}")
    print(f"  macro F1           {m['macro_f1']:.4f}")
    print(f"  per-class AUC      {m['per_class_auc']}")
    print(f"  per-class recall   {m['per_class_recall']}")
    print()
gap = meta.get("generalisation_gap", {})
print(f"generalisation gap (train acc at best epoch - split accuracy): {gap}")

## Per cohort

Only meaningful when the run pooled cohorts. The authors report AUC 0.78 on
I-SPY2 against 0.54 on DUKE for the same model — a 0.24 spread that a single
overall number hides completely.

In [ ]:
pred = pd.read_csv(RUN / "predictions_test.csv")
if "cohort" in pred.columns and pred.cohort.nunique() > 1:
    from sklearn.metrics import roc_auc_score
    names = cfg["class_names"]
    prob_cols = [f"prob_{n}" for n in names]
    rows = []
    for c, g in pred.groupby("cohort"):
        if g.label.nunique() < 2:
            continue
        p = g[prob_cols].to_numpy()
        auc = (roc_auc_score(g.label, p[:, 1]) if len(names) == 2
               else roc_auc_score(g.label, p, multi_class="ovr", average="macro"))
        rows.append({"cohort": c, "n": len(g), "auc": round(auc, 4),
                     "accuracy": round(g.correct.mean(), 4)})
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("single cohort — no breakdown to make")

## The source probe

The check that must run before any pooled result is believed. It trains the
identical pipeline with the COHORT as the label. Read it like this:

| probe macro-AUC | meaning |
|---|---|
| ≥ 0.90 | the images identify the cohort trivially — the result is contaminated |
| ~0.70 | a signature exists but does not dominate; report it beside the result |
| ~0.50 | cohorts are indistinguishable; pooling is safe |

Measured on the pooled three-cohort dataset: **0.9978**.

In [ ]:
# Uncomment to run it. It is a full training run, so it costs GPU time.
# from core.training import run
# probe = Config(pipeline=cfg["pipeline"], task=cfg["task"], model="resnet18",
#                cohorts=tuple(cfg["cohorts"]))
# run(probe, suffix="sourceprobe")
print("see docs/ for the recorded probe result: 0.9978 on the pooled dataset")

## Where it went wrong

In [ ]:
wrong = pred[~pred.correct]
print(f"{len(wrong)} of {len(pred)} patients misclassified\n")
print(pd.crosstab(pred.label_name, pred.pred_name, margins=True).to_string())
print("\nmost confident mistakes:")
names = cfg["class_names"]
wrong = wrong.assign(confidence=wrong[[f"prob_{n}" for n in names]].max(axis=1))
print(wrong.nlargest(8, "confidence")[
    ["pid", "label_name", "pred_name", "confidence"]].to_string(index=False))